In [3]:
import warnings

import numpy as np
import pandas as pd
import torch
from safetensors.torch import load_file
from tqdm import tqdm
from transformers import AutoModel

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

CACHE_DIR = "/data/users/neob"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

tumor_df = pd.read_csv("/home/neob/ai/paper/data/eval/tumor_type_eval.csv")
mgmt_df = pd.read_csv("/home/neob/ai/paper/data/eval/mgmt_eval.csv")
os_df = pd.read_csv("/home/neob/ai/paper/data/eval/os_eval.csv")

# Jinav3


In [6]:
def jina_embed(
    df: pd.DataFrame,
    output_path: str,
    text_col: str = "report",
    task: str = None,
    checkpoint_path: str = None,
    batch_size: int = 32,
    cache_dir: str = "/data/users/neob",
    device: str = None,
) -> np.ndarray:
    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    model = AutoModel.from_pretrained(
        "jinaai/jina-embeddings-v3",
        trust_remote_code=True,
        cache_dir=cache_dir,
    ).to(device)

    if checkpoint_path:
        checkpoint = load_file(checkpoint_path)
        model.load_state_dict(checkpoint, strict=False)

    model.eval()

    embeddings_list = []

    for i in tqdm(range(0, len(df), batch_size)):
        batch_texts = df[text_col].iloc[i : i + batch_size].tolist()

        with torch.no_grad():
            encode_kwargs = {
                "max_length": 2048,
                "device": device,
                "show_progress_bar": False,
            }
            if task:
                encode_kwargs["task"] = task

            embeddings = model.encode(batch_texts, **encode_kwargs)
            embeddings_list.append(embeddings)

    embeddings = np.vstack(embeddings_list)
    np.save(output_path, embeddings)

    return embeddings


## base model
jina_embed(
    df=os_df,
    task="text-matching",
    text_col="report",
    output_path="os/jina_base_text_match.npy",
    device=device,
)

## finetuned versions

jina_embed(
    df=os_df,
    task="text-matching",
    text_col="report",
    output_path="os/mnrl_cached.npy",
    checkpoint_path="/home/neob/ai/paper/models/models/mnrl_cached_full_data/final/model.safetensors",  # add finetuned checkpoint
    device=device,
)

# tumor type
jina_embed(
    df=tumor_df,
    task="text-matching",
    output_path="tumor/mnrl_full_data.npy",
    checkpoint_path="/home/neob/ai/paper/models/models/mnrl_cached_full_data/final/model.safetensors",
    device=device,
)

# mgmt
jina_embed(
    df=mgmt_df,
    task="text-matching",
    output_path="mgmt/mnrl_full_data.npy",
    checkpoint_path="/home/neob/ai/paper/models/models/mnrl_cached_full_data/final/model.safetensors",
    device=device,
)


100%|██████████| 17/17 [00:04<00:00,  3.96it/s]


array([[ 0.11983531, -0.04429702,  0.01048562, ...,  0.00053934,
         0.03478365, -0.02793734],
       [ 0.0812244 , -0.0606775 , -0.02724049, ...,  0.00737782,
         0.01274782, -0.02397902],
       [ 0.08436554, -0.05465848, -0.07296465, ..., -0.01437647,
         0.01013483, -0.00121582],
       ...,
       [ 0.10221114, -0.05326455, -0.00860325, ..., -0.01729263,
         0.0213884 , -0.01520047],
       [ 0.08018268, -0.05714071,  0.01097201, ..., -0.00447269,
        -0.0096658 , -0.02457599],
       [ 0.13416754, -0.01198961,  0.01174785, ..., -0.01778003,
        -0.00990791, -0.02041707]], dtype=float32)

# nv-embed


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import DataParallel
from tqdm import tqdm
from transformers import AutoModel


def nv_embed(
    df: pd.DataFrame,
    output_path: str,
    text_col: str = "report",
    instruction: str = "",
    batch_size: int = 16,
    cache_dir: str = "/data/users/neob",
    device_ids: list = None,
    max_length: int = 32768,
) -> np.ndarray:
    if device_ids is None:
        device_ids = [0, 1, 2, 3]

    model = AutoModel.from_pretrained(
        "nvidia/NV-Embed-v2", trust_remote_code=True, cache_dir=cache_dir
    )

    for module_key, module in model._modules.items():
        model._modules[module_key] = DataParallel(module, device_ids=device_ids)

    model.to(f"cuda:{device_ids[0]}")  # Primary GPU

    instruction_templates = {
        "none": "",
        "medical": """Instruct: Encode medical reports and capture key diagnostic findings, disease characteristics, and prognostic indicators.
Query: """,
    }

    instruction_prefix = instruction_templates.get(instruction, "")

    embeddings = []

    for i in tqdm(range(0, len(df), batch_size)):
        batch = df[text_col].iloc[i : i + batch_size].tolist()

        with torch.no_grad():
            batch_embeddings = model.encode(
                batch,
                instruction=instruction_prefix,
                max_length=max_length,
                show_progress_bar=False,
            )

            # Normalize embeddings
            batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
            embeddings.append(batch_embeddings)

    all_embeddings = torch.cat(embeddings, dim=0)

    # Verify embedding dimensions
    assert len(all_embeddings) == len(df), (
        "Embeddings length doesn't match dataframe length"
    )

    # Save embeddings
    np.save(output_path, all_embeddings.cpu().numpy())

    return all_embeddings.cpu().numpy()


# Overall survival with medical instruction
nv_embed(
    df=os_df,
    instruction="medical",  # or None
    output_path="os/nv_embed_medical.npy",
    device_ids=[0, 1, 2, 3],  # set GPU IDs for embedding generation
)

# Tumor type
nv_embed(
    df=tumor_df,
    instruction="none",
    output_path="tumor_type/nv_embed.npy",
    device_ids=[4, 5, 6, 7],
)

# MGMT
nv_embed(
    df=mgmt_df,
    instruction="none",
    output_path="mgmt/nv_embed_generic.npy",
    device_ids=[0, 1, 2, 3],
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 280/280 [12:04<00:00,  2.59s/it]


First few MRNs in dataframe: [10183421, 10183421, 10183421, 10183421, 10183421]
Shape of embeddings: torch.Size([4470, 4096])


# Gatortron


In [7]:
def gatortron_embed(
    df: pd.DataFrame,
    output_path: str,
    text_col: str = "report",
    checkpoint_path: str = None,
    batch_size: int = 32,
    max_length: int = 512,  # Gatortron default max length
    cache_dir: str = "/data/users/neob",
    device: str = None,
) -> np.ndarray:
    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    # Load the model and tokenizer separately
    from transformers import AutoConfig, AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        "UFNLP/gatortron-large",
        trust_remote_code=True,
        cache_dir=cache_dir,
    )

    config = AutoConfig.from_pretrained(
        "UFNLP/gatortron-large",
        trust_remote_code=True,
        cache_dir=cache_dir,
    )

    model = AutoModel.from_pretrained(
        "UFNLP/gatortron-large",
        trust_remote_code=True,
        cache_dir=cache_dir,
        config=config,
    ).to(device)

    if checkpoint_path:
        checkpoint = load_file(checkpoint_path)
        model.load_state_dict(checkpoint, strict=False)

    model.eval()

    embeddings_list = []

    # Process batches
    for i in tqdm(range(0, len(df), batch_size)):
        batch_texts = df[text_col].iloc[i : i + batch_size].tolist()

        # Get embeddings
        with torch.no_grad():
            # Tokenize the batch using separate tokenizer
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)

            # Get model outputs
            outputs = model(**inputs)

            # Use CLS token embeddings (first token)
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings_list.append(embeddings)

    # Stack all embeddings
    embeddings = np.vstack(embeddings_list)
    np.save(output_path, embeddings)

    return embeddings


# Overall survival
gatortron_embed(
    df=os_df,
    text_col="report",
    output_path="os/gatortron_large.npy",
    device="cuda:7",
)

# Tumor type
gatortron_embed(
    df=tumor_df,
    text_col="report",
    output_path="tumor_type/gatortron_large.npy",
    device="cuda:6",
)

# MGMT
gatortron_embed(
    df=mgmt_df,
    text_col="report",
    output_path="mgmt/gatortron_large.npy",
    device="cuda:5",
)

100%|██████████| 17/17 [04:43<00:00, 16.67s/it]


array([[-0.35874695, -0.10331648,  0.1115647 , ..., -0.18005404,
        -0.2706062 ,  0.17075016],
       [-0.17964222, -0.18605341, -0.22009447, ..., -0.20132647,
        -0.11884459, -0.18112661],
       [-0.4462781 , -0.13722673,  0.03627316, ..., -0.07853854,
        -0.26773432,  0.1363074 ],
       ...,
       [-0.35320532, -0.15935303,  0.00320681, ..., -0.13154063,
        -0.14650208,  0.11385939],
       [-0.40986878, -0.06830108, -0.15080534, ..., -0.22722155,
        -0.26989052,  0.0739908 ],
       [-0.0999138 , -0.13787493,  0.06963445, ..., -0.31532347,
        -0.25338677,  0.00358049]], dtype=float32)